In [57]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow import keras

from art.attacks.evasion import FastGradientMethod, DeepFool
from art.estimators.classification import TensorFlowV2Classifier

In [58]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32")  / 255.0
y_train = y_train.flatten()
y_test  = y_test.flatten()

In [52]:
model = keras.models.load_model("model.keras")

classifier = TensorFlowV2Classifier(
    model=model,
    loss_object=tf.keras.losses.CategoricalCrossentropy(from_logits=False),
    nb_classes=10,
    input_shape=(32, 32, 3),
    clip_values=(0, 1),
)

In [53]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()

def fgsm_attack(model, images, labels, eps=8/255):
    images = tf.cast(images, tf.float32)
    labels = tf.cast(labels, tf.int32)

    with tf.GradientTape() as tape:
        tape.watch(images)
        predictions = model(images, training=False)
        loss = loss_fn(labels, predictions)

    grad       = tape.gradient(loss, images)
    adv_images = images + eps * tf.sign(grad)
    adv_images = tf.clip_by_value(adv_images, 0.0, 1.0)
    return adv_images

In [54]:
EPS = 8 / 255
N   = 1000

x_sample = x_test[:N]
y_sample  = y_test[:N]
y_onehot  = tf.keras.utils.to_categorical(y_sample, num_classes=10)

# baseline
clean_preds = np.argmax(classifier.predict(x_sample), axis=1)
clean_acc   = np.mean(clean_preds == y_sample)

# art fgsm
art_attack = FastGradientMethod(estimator=classifier, eps=EPS)
x_adv_art  = art_attack.generate(x=x_sample, y=y_onehot)
art_preds  = np.argmax(classifier.predict(x_adv_art), axis=1)
art_acc    = np.mean(art_preds == y_sample)

# fgsm
x_adv_own = fgsm_attack(model, x_sample, y_sample, eps=EPS).numpy()
own_preds = np.argmax(classifier.predict(x_adv_own), axis=1)
own_acc   = np.mean(own_preds == y_sample)

# DeepFool
df_attack      = DeepFool(classifier=classifier, max_iter=50)
x_adv_deepfool = df_attack.generate(x=x_sample)
df_preds       = np.argmax(classifier.predict(x_adv_deepfool), axis=1)
df_acc         = np.mean(df_preds == y_sample)

print(f"Clean accuracy      : {clean_acc:.4f}")
print(f"ART FGSM accuracy   : {art_acc:.4f}  (eps={EPS:.4f})")
print(f"Own FGSM accuracy   : {own_acc:.4f}  (eps={EPS:.4f})")
print(f"DeepFool accuracy   : {df_acc:.4f}")

DeepFool:   0%|          | 0/1000 [00:00<?, ?it/s]

Clean accuracy      : 0.8590
ART FGSM accuracy   : 0.1820  (eps=0.0314)
Own FGSM accuracy   : 0.1820  (eps=0.0314)
DeepFool accuracy   : 0.1640


In [59]:
from art.attacks.evasion import UniversalPerturbation

N_TRAIN = 2000
x_train_sample = x_train[:N_TRAIN]

uap = UniversalPerturbation(
    classifier=classifier,
    attacker="fgsm",
    eps=8/255,
    delta=0.2,             # docelowy fooling rate (20%)
    max_iter=10,
)
uap.generate(x=x_train_sample)
noise = uap.noise # jeden szum na caly zbior

# Aplikuj na zbiór testowy
x_adv_uap = np.clip(x_sample + noise, 0.0, 1.0)
uap_preds = np.argmax(classifier.predict(x_adv_uap), axis=1)
uap_acc   = np.mean(uap_preds == y_sample)

print(f"Clean accuracy : {clean_acc:.4f}")
print(f"UAP accuracy   : {uap_acc:.4f}")
print(f"Fooling rate   : {1 - uap_acc:.4f}")
print(f"Noise L-inf    : {np.abs(noise).max():.4f}")

Universal perturbation:   0%|          | 0/10 [00:00<?, ?it/s]

Clean accuracy : 0.8590
UAP accuracy   : 0.6180
Fooling rate   : 0.3820
Noise L-inf    : 0.0314
